In [1]:
%pip install -U langmem


   ------------- -------------------------- 1/3 [trustcall]
   -------------------------- ------------- 2/3 [langmem]
   -------------------------- ------------- 2/3 [langmem]
   -------------------------- ------------- 2/3 [langmem]
   -------------------------- ------------- 2/3 [langmem]
   -------------------------- ------------- 2/3 [langmem]
   ---------------------------------------- 3/3 [langmem]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langmem import create_manage_memory_tool, create_search_memory_tool
from dotenv import load_dotenv
import os


#set the OpenAI API key
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

c:\Users\sam\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
# Import core components 
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langmem import create_manage_memory_tool, create_search_memory_tool

# Set up storage 
store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
) 

# Create an agent with memory capabilities 
agent = create_react_agent(
    "gpt-3.5-turbo-0125",
    tools=[
        create_manage_memory_tool(namespace=("memories",)),
        create_search_memory_tool(namespace=("memories",)),
    ],
    store=store,
)

C:\Users\sam\AppData\Local\Temp\ipykernel_15492\1886586169.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [4]:
response=agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Kathan and I like Porsche 911"}]}
)
print(response['messages'][-1].content)

##OUTPUT
# Great to meet you, Kathan! I've noted that you like Porsche 911. If there's anything specific you'd like to know or discuss about Porsche 911, feel free to ask!

Great to know, Kathan! I've made a note that you like Porsche 911. If there's anything else you'd like me to remember or assist you with, feel free to let me know!


In [5]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me about myself with my name."}]}
)
print(response["messages"][-1].content)

##OUTPUT
# Based on the information I have in memory, your name is Kathan, and you like Porsche 911. If you would like to share more about yourself, feel free to do so!

NumPy not found in the current Python environment. The InMemoryStore will use a pure Python implementation for vector operations, which may significantly impact performance, especially for large datasets or frequent searches. For optimal speed and efficiency, consider installing NumPy: pip install numpy


I remember that your name is Kathan, and you like Porsche 911. If there's anything else you'd like me to know, feel free to share!


In [ ]:
###HOT PATH###
https://langchain-ai.github.io/langmem/hot_path_quickstart/#agent
https://dragonforest.in/langmem-begginers-guide/

In [6]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore
from langgraph.utils.config import get_store 
from langmem import create_manage_memory_tool

def prompt(state):
    """Prepare the messages for the LLM."""
    store = get_store() #
    memories = store.search(
        ("memories",),
        query=state["messages"][-1].content,
    )
    system_msg = f"""You are a helpful assistant.

## Memories
<memories>
{memories}
</memories>
"""
    return [{"role": "system", "content": system_msg}, *state["messages"]]

store = InMemoryStore(
    index={ # Store extracted memories 
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
) 
checkpointer = MemorySaver() # Checkpoint for  graph state 

agent = create_react_agent( 
    "gpt-3.5-turbo-0125",
    prompt=prompt,
    tools=[create_manage_memory_tool(namespace=("memories",))],
    store=store,
    checkpointer=checkpointer, 
)

C:\Users\sam\AppData\Local\Temp\ipykernel_15492\2570701830.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [9]:
config = {"configurable": {"thread_id": "thread-a"}}
response=agent.invoke({"messages":[{"role":"user","content":"My favorite car is Porsche 911."}]},
              config= config)
              
print(response['messages'][-1].content)
##OUTPUT
#I've noted that your favorite car is the Porsche 911. If you need me to remember anything else or if you have any other preferences, feel free to let me know!

I've made a note that your favorite car is Porsche 911. If there's anything else you'd like me to remember or assist you with, feel free to let me know!


In [11]:
response=agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Which is my favorite car?"}
        ]
    },
    config=config,
)

print(response['messages'][-1].content)
##OUTPUT
# Your favorite car is the Porsche 911

Your favorite car is the Porsche 911.


In [12]:
#In the background method
#In this method, the memories will be stored in the background without any delay in the flow. Let’s check it out.

from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint
from langgraph.store.memory import InMemoryStore
from langmem import ReflectionExecutor, create_memory_store_manager
store = InMemoryStore( # 
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
)  
llm = init_chat_model("gpt-3.5-turbo-0125")
memory_manager = create_memory_store_manager(
    "gpt-3.5-turbo-0125",
    namespace=("memories",),  # 
)
@entrypoint(store=store)  # Create a LangGraph workflow
async def chat(message: str):
    response = llm.invoke(message)
    to_process = {"messages": [{"role": "user", "content": message}] + [response]}
    await memory_manager.ainvoke(to_process)  # 
    return response.content
# Run conversation as normal
response = await chat.ainvoke(
    "I like car. My favorite car is Porsche 911.",
)
print(response)

The Porsche 911 is a classic sports car with a rich history and a reputation for performance and elegance. What do you like most about the Porsche 911?
